In [0]:
%run "../00_config"

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime, timezone

##Load Bronze Orders

In [0]:
df_bronze = spark.table(BRONZE_ORDERS_ENRICHED)
display(df_bronze)

##Clean and transform

In [0]:
df_silver = (df_bronze
    # Cast types
    .withColumn("unit_price_sar",    F.col("unit_price_sar").cast(DoubleType()))
    .withColumn("quantity",          F.col("quantity").cast(IntegerType()))

    # Parse order_date to DateType
    .withColumn("order_date",
        F.to_date(F.col("order_date"), "dd/MM/yyyy")
    )

    # Replace empty strings with null
    .withColumn("customer_id",
        F.when(F.col("customer_id") == "", F.lit(None))
        .otherwise(F.col("customer_id"))
    )
    .withColumn("product_id",
        F.when(F.col("product_id") == "", F.lit(None))
        .otherwise(F.col("product_id"))
    )
    .withColumn("city",
        F.when(F.col("city") == "", F.lit(None))
        .otherwise(F.col("city"))
    )
    .withColumn("customer_name",
        F.when(F.col("customer_name") == "", F.lit(None))
        .otherwise(F.col("customer_name"))
    )
    .withColumn("device_type",
        F.when(F.col("device_type") == "", F.lit(None))
        .otherwise(F.col("device_type"))
    )
    .withColumn("order_status",
        F.when(F.col("order_status") == "", F.lit(None))
        .otherwise(F.col("order_status"))
    )
    .withColumn("payment_method",
        F.when(F.col("payment_method") == "", F.lit(None))
        .otherwise(F.col("payment_method"))
    )

     # Standardize order_status
    .withColumn("city",
        F.initcap(F.col("city"))
    )
    .withColumn("order_status",
        F.initcap(F.col("order_status"))
    )
    # Standardize payment_method
    .withColumn("payment_method",
        F.initcap(F.col("payment_method"))
    )
    # Standardize device_type
    .withColumn("device_type",
        F.initcap(F.col("device_type"))
    )

    # Clean date
    .withColumn("_snapshot_date", (F.to_date(F.col("_snapshot_date"), "dd-MM-yyyy")))

    # Filter out orders with no order_id, quantity and unit_price_sar
    .filter(F.col("order_id") != "")
    .filter(F.col("order_id").isNotNull())
    .filter(F.col("quantity") > 0)
    .filter(F.col("quantity").isNotNull())
    .filter(F.col("unit_price_sar") > 0)
    .filter(F.col("unit_price_sar").isNotNull())
    
    # Deduplicate — keep one row per order
    .dropDuplicates(["order_id"])

    # Add ingestion metadata
    .withColumn("_silver_ingested_at", F.to_timestamp(F.lit(datetime.now(timezone.utc).isoformat())))

    # Final column selection
    .select(
        "order_id",
        "customer_id",
        "product_id",
        "order_date",
        "quantity",
        "unit_price_sar",
        "order_status",
        "customer_name",
        "city",
        "device_type",
        "payment_method",
        "_snapshot_date",
        "_silver_ingested_at"
    )
)

print(f"Silver rows after cleaning: {df_silver.count()}")
display(df_silver)

##Write to silver

In [0]:
(df_silver
   .write
   .format("delta")
   .mode("overwrite")
   .saveAsTable(SILVER_ORDERS)
)
print(f"✅ {df_silver.count()} orders written to {SILVER_ORDERS}")

##Validate

In [0]:
df_check = spark.table(SILVER_ORDERS)
print(f"Total rows        : {df_check.count()}")
print(f"Distinct orders   : {df_check.select('order_id').distinct().count()}")
print(f"Cities            : {[r[0] for r in df_check.select('city').distinct().collect()]}")
print(f"Order statuses    : {[r[0] for r in df_check.select('order_status').distinct().collect()]}")
df_check.printSchema()
df_check.select(
   "order_id", "city", "product_id",
   "quantity", "unit_price_sar", "order_status"
).show(5, truncate=35)

In [0]:
%sql
SELECT * FROM saudi_retail_catalog.silver.silver_orders;